# Real-Time Dataset Generation using SDXL Turbo
This notebook demonstrates how to generate a custom image dataset (e.g., for Jigsaw Puzzles) in real-time using **Stable Diffusion XL Turbo** (SDXL Turbo).

### Hardware Requirements
To run this generation efficiently, make sure your Colab runtime is set to use a **GPU** (e.g., T4):


### Step 1: Mount Google Drive
Mount your Google Drive to save the generated images directly to your cloud storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Install Required Libraries
Install `diffusers` for the model pipeline, along with `transformers`, `accelerate`, and `safetensors` for optimization and loading.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

### Step 3: Initialize the SDXL Turbo Pipeline
Load the pre-trained SDXL Turbo weights using half-precision to save VRAM and move the execution pipeline to the GPU (`cuda`). Also enable attention slicing to prevent out-of-memory errors.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16
)

pipe.to("cuda")
pipe.enable_attention_slicing()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

### Step 4: Configure the Prompt Style & Scenes

Below is the prompt structure used to generate a consistent, stylized dataset.

* **Style:** Low-poly 3D render, isometric view, warm flat shading.
* **Customization:** You can modify the `BASE_PROMPT` style or alter the list of `scenes`. Adjusting these settings will not affect the generation performance.

In [ ]:
BASE_PROMPT = """
A {scene}, low-poly 3D render style, isometric view,
soft warm lighting from upper left, vibrant saturated colors,
clean flat shading with sharp polygon edges, minimal shadows,
white or light blue solid background, centered composition with
full subject visible, no text or watermarks,
square format, high resolution
"""

In [ ]:
scenes = [
"a minimalist bedroom with a bed and a single nightstand",
"a sunlit living room with large windows and curtains",
"a compact kitchen with hanging utensils",
"a hallway with doors on both sides",
"a staircase inside a modern house",
"a bathroom with a bathtub and towels",
"a home office with dual monitors",
"a dining area with a round table",
"a child's bedroom with a bunk bed",
"a wardrobe closet with open doors",
"a scene inspired by a surreal dream with floating objects",
"a scene inspired by impressionist painting with soft brush-like textures",
"a scene inspired by cubism with fragmented shapes",
"a scene inspired by abstract geometric art",
"a scene inspired by renaissance interior lighting",
"a scene inspired by baroque dramatic lighting",
"a scene inspired by minimalism with empty space",
"a scene inspired by pop art with bold colors",
"a scene inspired by expressionism with distorted forms",
"a scene inspired by modern digital illustration",
"a stylized cartoon jungle with animals",
"a stylized cartoon underwater world",
"a stylized cartoon mountain landscape",
"a stylized cartoon desert with cacti",
"a stylized cartoon snowy village",
"a stylized cartoon space scene with planets",
"a stylized cartoon farm with animals",
"a stylized cartoon amusement park",
"a stylized cartoon city skyline",
"a stylized cartoon island with palm trees",
"a cinematic scene of a lone traveler in a vast desert",
"a cinematic scene of a storm approaching a small house",
"a cinematic scene of a boat in the middle of the ocean",
"a cinematic scene of a person standing under street lights",
"a cinematic scene of a forest with rays of light",
"a cinematic scene of a mountain expedition",
"a cinematic scene of a road stretching into the horizon",
"a cinematic scene of a bridge in heavy fog",
"a cinematic scene of a cabin during snowfall",
"a cinematic scene of a city during rain",
"a panda sitting and eating bamboo",
"a lion resting under a tree",
"a tiger walking in tall grass",
"a penguin standing on ice",
"a dolphin jumping out of water",
"a turtle near the shore",
"a parrot sitting on a perch",
"a hamster inside a cage",
"a goat standing on rocks",
"a duck swimming in a pond",
"a person jogging in a park",
"a person typing on a keyboard",
"a person talking on the phone",
"a person sitting on the floor reading",
"a person carrying a backpack",
"a person waiting at a bus stop",
"a person crossing a street",
"a person standing in front of a mirror",
"a person opening a door",
"a person looking at a map",
"a tall office tower with reflective windows",
"a small shop on a street corner",
"a residential apartment building",
"a lighthouse near calm water",
"a windmill on a grassy field",
"a tower with a clock face",
"a castle wall with a gate",
"a greenhouse made of glass",
"a train station building",
"a fire station with a garage",
"a single chair placed in an empty room",
"a desk with a laptop and notebook",
"a shelf with neatly arranged books",
"a lamp on a bedside table",
"a backpack placed on the ground",
"a pair of sneakers on the floor",
"a bicycle leaning against a wall",
"a suitcase standing upright",
"a trash bin in a corner",
"a potted plant near a window",
"a picnic blanket with food items",
"a campfire with logs around it",
"a fishing rod by the water",
"a barbecue setup with grill",
"a garden path with stepping stones",
"a bench facing a scenic view",
"a minimalist white office desk with a laptop and notebook",
"a cozy attic bedroom with slanted ceiling and window",
"a luxury bathroom with a bathtub and candles",
"a kitchen island with stools and hanging lights",
"a cluttered workshop with scattered tools",
"a bright nursery room with a crib and toys",
"a home library with a ladder and tall shelves",
"a modern dining room with a chandelier",
"a small balcony with plants and chairs",
"a hallway with framed pictures on walls",
"a scene inspired by a cubist painting with fragmented shapes",
"a scene inspired by impressionist style with soft brush-like textures",
"a scene inspired by surrealism with floating objects",
"a scene inspired by abstract geometric composition",
"a scene inspired by renaissance interior with symmetry",
"a scene inspired by baroque dramatic lighting",
"a scene inspired by minimal art with simple forms",
"a scene inspired by pop art with bold colors",
"a scene inspired by ukiyo-e style landscape",
"a scene inspired by expressionism with distorted perspective",
"a cartoon-style jungle with oversized leaves",
"a cartoon-style desert with simplified dunes",
"a cartoon-style snow village with rounded houses",
"a cartoon-style underwater world with fish and coral",
"a cartoon-style mountain landscape with soft peaks",
"a cartoon-style farm with bright animals",
"a cartoon-style airport with simple planes",
"a cartoon-style harbor with tiny boats",
"a cartoon-style amusement park with rides",
"a cartoon-style castle with flags",
"a dramatic scene of a lone figure in a vast desert",
"a peaceful lakeside cabin reflection scene",
"a busy urban crosswalk with people walking",
"a quiet subway interior with empty seats",
"a cinematic forest path with light rays",
"a scene of a storm approaching a small house",
"a traveler standing at a mountain viewpoint",
"a foggy harbor with boats barely visible",
"a city skyline viewed from a rooftop",
"a rural dirt road stretching into the distance",
"a penguin standing on ice",
"a lion resting under a tree",
"a tiger walking through grass",
"a monkey hanging from a branch",
"a bear near a river",
"a wolf standing on a hill",
"a elephant walking across a plain",
"a giraffe eating leaves from a tree",
"a zebra standing in a field",
"a kangaroo in an open landscape",
"a person typing on a keyboard",
"a person stretching in a yoga pose",
"a person sitting on stairs thinking",
"a person taking a photo with a camera",
"a person carrying a backpack walking",
"a person sitting in a café using a laptop",
"a person watering plants indoors",
"a person opening a door",
"a person tying their shoes",
"a person looking at a map",
"a modern apartment building exterior",
"a rustic farmhouse with a fence",
"a tall radio tower structure",
"a small roadside shop",
"a fire station building with garage doors",
"a police station exterior",
"a gas station with pumps",
"a train depot building",
"a lighthouse with surrounding buildings",
"a greenhouse made of glass panels",
"a close-up of a desk lamp",
"a close-up of a wristwatch",
"a close-up of a pair of glasses",
"a close-up of a smartphone",
"a close-up of a notebook",
"a close-up of a pen and paper",
"a close-up of a coffee mug",
"a close-up of headphones",
"a close-up of a plant leaf",
"a close-up of a door handle",
"a picnic basket open on grass",
"a plate of fruits arranged neatly",
"a stack of folded towels",
"a set of cooking pots on a stove",
"a toolbox with compartments open",
"a pile of books on a table",
"a shoe rack with multiple pairs",
"a coat hanging on a rack",
"a backpack placed on the floor",
"a suitcase packed with clothes",
"a playground swing set",
"a park pathway lined with trees",
"a street corner with crosswalk",
"a bus driving along a road",
"a bicycle leaning against a wall",
"a row of mailboxes",
"a street café outdoor seating",
"a small public square with benches",
"a parking garage interior",
"a roadside billboard structure",
"a cozy reading corner with a chair, lamp, and bookshelf",
"a messy teenager bedroom with posters and a desk",
"a modern office meeting room with a large table",
"a small bathroom with a mirror and sink",
"a kitchen countertop with fruits and utensils",
"a dining table set for dinner",
"a laundry room with a washing machine",
"a garage with tools and a workbench",
"a studio apartment with multifunctional furniture",
"a child's playroom with toys scattered",
"a scene inspired by The Starry Night swirling sky over a small town",
"a scene inspired by The Persistence of Memory with melting clocks",
"a scene inspired by The Scream with a figure on a bridge",
"a scene inspired by Girl with a Pearl Earring in a simple room",
"a scene inspired by American Gothic with two figures and a house",
"a scene inspired by The Last Supper with a long dining table",
"a scene inspired by Water Lilies with a pond and reflections",
"a scene inspired by The Birth of Venus with a figure on water",
"a scene inspired by The Night Watch with a group of people",
"a scene inspired by The Kiss with two figures embracing",
"a cartoon-style suburban house with bright colors",
"a cartoon-style park with exaggerated trees",
"a cartoon-style classroom with oversized furniture",
"a cartoon-style kitchen with playful proportions",
"a cartoon-style city street with simplified cars",
"a cartoon-style beach with stylized waves",
"a cartoon-style forest with rounded trees",
"a cartoon-style playground with vibrant equipment",
"a cartoon-style shop with colorful signage",
"a cartoon-style bedroom with soft shapes",
"a dramatic spaceship interior corridor scene",
"a heroic figure standing on a cliff overlooking a vast land",
"a chase scene on a city street with cars",
"a quiet scene of a person looking out a rainy window",
"a marketplace scene full of activity",
"a train station platform with travelers waiting",
"a rooftop scene overlooking a glowing city",
"a desert journey with a traveler and vehicle",
"a mysterious alleyway with dim lighting",
"a group of people around a campfire at night",
"a cat sitting on a chair",
"a dog playing with a ball",
"a bird perched on a branch",
"a fish swimming in a bowl",
"a horse standing in a field",
"a cow grazing on grass",
"a sheep in a fenced area",
"a rabbit near a small hut",
"a fox in a forest clearing",
"a deer near a lake",
"a person sitting at a desk working on a laptop",
"a person reading a book on a couch",
"a person cooking in a kitchen",
"a person painting on a canvas",
"a person exercising with dumbbells",
"a person riding a bicycle",
"a person walking a dog",
"a person drinking coffee at a table",
"a person using a smartphone",
"a person gardening with tools",
"a tall skyscraper building",
"a small suburban house with a garage",
"a wooden cabin in the woods",
"a glass office building",
"a factory with chimneys",
"a school building with a playground",
"a hospital building with an entrance",
"a church with a tall tower",
"a shopping mall exterior",
"a warehouse building",
"a close-up of a chair",
"a close-up of a table",
"a close-up of a lamp",
"a close-up of a computer monitor",
"a close-up of a keyboard",
"a close-up of a backpack",
"a close-up of a pair of shoes",
"a close-up of a water bottle",
"a close-up of a bicycle",
"a close-up of a suitcase",
"a fruit bowl with apples and bananas",
"a plate with food on a table",
"a cup of coffee on a desk",
"a teapot with cups",
"a stack of books",
"a pile of clothes",
"a set of tools arranged neatly",
"a group of boxes stacked",
"a collection of plants in pots",
"a basket with groceries",
"a playground slide",
"a park bench under a tree",
"a street lamp at night",
"a mailbox on a post",
"a traffic light at an intersection",
"a bus stop shelter",
"a vending machine",
"a public trash bin",
"a cozy reading corner with a chair, lamp, and bookshelf",
"a messy teenager bedroom with posters and a desk",
"a modern office meeting room with a large table",
"a small bathroom with a mirror and sink",
"a kitchen countertop with fruits and utensils",
"a dining table set for dinner",
"a laundry room with a washing machine",
"a garage with tools and a workbench",
"a studio apartment with multifunctional furniture",
"a child's playroom with toys scattered",
"a scene inspired by The Starry Night swirling sky over a small town",
"a scene inspired by The Persistence of Memory with melting clocks",
"a scene inspired by The Scream with a figure on a bridge",
"a scene inspired by Girl with a Pearl Earring in a simple room",
"a scene inspired by American Gothic with two figures and a house",
"a scene inspired by The Last Supper with a long dining table",
"a scene inspired by Water Lilies with a pond and reflections",
"a scene inspired by The Birth of Venus with a figure on water",
"a scene inspired by The Night Watch with a group of people",
"a scene inspired by The Kiss with two figures embracing",
"a cartoon-style suburban house with bright colors",
"a cartoon-style park with exaggerated trees",
"a cartoon-style classroom with oversized furniture",
"a cartoon-style kitchen with playful proportions",
"a cartoon-style city street with simplified cars",
"a cartoon-style beach with stylized waves",
"a cartoon-style forest with rounded trees",
"a cartoon-style playground with vibrant equipment",
"a cartoon-style shop with colorful signage",
"a cartoon-style bedroom with soft shapes",
"a dramatic spaceship interior corridor scene",
"a heroic figure standing on a cliff overlooking a vast land",
"a chase scene on a city street with cars",
"a quiet scene of a person looking out a rainy window",
"a marketplace scene full of activity",
"a train station platform with travelers waiting",
"a rooftop scene overlooking a glowing city",
"a desert journey with a traveler and vehicle",
"a mysterious alleyway with dim lighting",
"a group of people around a campfire at night",
"a cat sitting on a chair",
"a dog playing with a ball",
"a bird perched on a branch",
"a fish swimming in a bowl",
"a horse standing in a field",
"a cow grazing on grass",
"a sheep in a fenced area",
"a rabbit near a small hut",
"a fox in a forest clearing",
"a deer near a lake",
"a person sitting at a desk working on a laptop",
"a person reading a book on a couch",
"a person cooking in a kitchen",
"a person painting on a canvas",
"a person exercising with dumbbells",
"a person riding a bicycle",
"a person walking a dog",
"a person drinking coffee at a table",
"a person using a smartphone",
"a person gardening with tools",
"a tall skyscraper building",
"a small suburban house with a garage",
"a wooden cabin in the woods",
"a glass office building",
"a factory with chimneys",
"a school building with a playground",
"a hospital building with an entrance",
"a church with a tall tower",
"a shopping mall exterior",
"a warehouse building",
"a close-up of a chair",
"a close-up of a table",
"a close-up of a lamp",
"a close-up of a computer monitor",
"a close-up of a keyboard",
"a close-up of a backpack",
"a close-up of a pair of shoes",
"a close-up of a water bottle",
"a close-up of a bicycle",
"a close-up of a suitcase",
"a fruit bowl with apples and bananas",
"a plate with food on a table",
"a cup of coffee on a desk",
"a teapot with cups",
"a stack of books",
"a pile of clothes",
"a set of tools arranged neatly",
"a group of boxes stacked",
"a collection of plants in pots",
"a basket with groceries",
"a playground slide",
"a park bench under a tree",
"a street lamp at night",
"a mailbox on a post",
"a traffic light at an intersection",
"a bus stop shelter",
"a vending machine",
"a public trash bin",
"a bicycle rack",
"a fountain in a plaza",
"a small island with a lighthouse",
"a floating island with waterfalls",
"a cozy cabin in a snowy forest",
"a desert oasis with palm trees",
"a futuristic city block with neon accents",
"a medieval village house",
"a tiny house on wheels",
"a treehouse village connected by bridges",
"a mountain peak with a small shrine",
"a coastal cliff with crashing waves",
"a peaceful Japanese garden with a pond",
"a park with a fountain and benches",
"a campsite with a tent and campfire",
"a farm with a red barn and windmill",
"a vineyard on rolling hills",
"a lighthouse on rocky terrain",
"a harbor with small boats",
"a bridge over a calm river",
"a snowy village street",
"a tropical beach with umbrellas",
"a cozy bedroom with plants",
"a modern living room with a sofa and TV",
"a minimalist workspace desk setup",
"a kitchen with cooking utensils",
"a library with tall bookshelves",
"a classroom with desks and a board",
"a small café interior with tables",
"a bakery display with pastries",
"a music studio with instruments",
"a gaming room with neon lights",
"a playground with slides and swings",
"a skate park with ramps",
"a basketball court",
"a soccer field with goal posts",
"a swimming pool area",
"a gym with workout equipment",
"a yoga studio",
"a running track",
"a small stadium",
"a park with a cycling path",
"a robot standing in a lab",
"a drone on a landing pad",
"a spaceship on a platform",
"a futuristic control room",
"a satellite dish station",
"a rover on a rocky planet",
"a sci-fi city rooftop",
"a charging station for robots",
"a mechanical workshop",
"a conveyor belt factory",
"a fantasy castle on a hill",
"a wizard tower with glowing windows",
"a dragon resting on rocks",
"a magical forest with glowing plants",
"a floating castle in the sky",
"a portal in a forest clearing",
"a treasure chest in a cave",
"a crystal cave with glowing gems",
"a ruined temple in the jungle",
"a giant mushroom village",
"a hot air balloon above land",
"a train passing through mountains",
"a bus stop with shelter",
"a small airport runway",
"a parking lot with cars",
"a bicycle parked near a tree",
"a highway intersection",
"a tunnel entrance in a hill",
"a ferry dock",
"a tram station",
"a market with small stalls",
"a street food stand",
"a flower shop exterior",
"a bookstore storefront",
"a clothing shop display",
"a supermarket aisle",
"a vending machine corner",
"a delivery truck unloading boxes",
"a warehouse storage area",
"a package sorting station",
"a rainy street with puddles",
"a snowy forest clearing",
"a sunny meadow with flowers",
"a foggy forest path",
"an autumn park with falling leaves",
"a night scene with street lights",
"a sunrise over hills",
"a sunset at the beach",
"a windy hill with trees bending",
"a cloudy sky over a town",
"a picnic setup on grass",
"a barbecue grill in a backyard",
"a birthday party table setup",
"a camping van near a lake",
"a fishing spot by a river",
"a dog house in a yard",
"a cat sitting on a windowsill",
"a playground sandbox",
"a garden with a watering can",
"a mailbox beside a country road",
"a cozy reading corner with a plush chair, warm lamp, and wooden bookshelf",
"a messy teenager bedroom with band posters, clothes on the floor, and a desk",
"a modern office meeting room with a large glass table and ergonomic chairs",
"a small minimalist bathroom with a circular mirror and a white porcelain sink",
"a sunlit kitchen countertop with a bowl of citrus fruits and scattered utensils",
"a misty pine forest at dawn with sunlight filtering through the needles",
"a bustling cyberpunk street market with neon signs and rain puddles",
"a tranquil Japanese zen garden with raked sand and a single bonsai tree",
"a medieval castle armory with rows of gleaming plate armor and swords",
"an abandoned Victorian greenhouse overgrown with tropical vines and ivy",
"a high-fantasy wizard's tower filled with floating books and glowing crystals",
"a futuristic space station corridor with sliding metallic doors and blue LED strips",
"a rustic cabin living room with a roaring stone fireplace and bearskin rug",
"a vibrant underwater coral reef with schools of tropical fish and sea anemones",
"a dusty wild west saloon with swinging wooden doors and a piano",
"a serene mountain lake reflecting a snow-capped peak at sunset",
"a crowded Moroccan bazaar with hanging lanterns and colorful spice piles",
"a sleek laboratory with bubbling beakers and holographic displays",
"a whimsical candy land with lollipop trees and a river of chocolate",
"a vintage 1950s diner with red vinyl booths and a jukebox",
"a spooky graveyard under a full moon with crooked tombstones and fog",
"a grand ballroom with gold-leaf pillars and a massive crystal chandelier",
"a post-apocalyptic city street with rusted cars and cracked asphalt",
"a traditional tea room with tatami mats and a low wooden table",
"a bright artist's studio with paint-splattered canvases and tall windows",
"a pirate ship deck with coiled ropes, a wooden helm, and tattered sails",
"a cozy hobbit hole with a round green door and a pantry full of cheese",
"a bustling Parisian sidewalk cafe with wicker chairs and flower boxes",
"a hidden cave grotto with bioluminescent mushrooms and a clear pond",
"a retro arcade with glowing cabinet screens and pixelated wall art",
"a serene lavender field in Provence with a small stone cottage",
"a high-tech cockpit of a giant mecha with complex control panels",
"a snowy village square with a decorated Christmas tree and glowing windows",
"an ancient Egyptian tomb chamber with hieroglyphics and a gold sarcophagus",
"a cluttered attic with old trunks, a rocking horse, and cobwebs",
"a rooftop garden overlooking a skyline with lounge chairs and potted palms",
"a magical forest clearing where the flowers glow in the dark",
"a steampunk workshop with brass gears, steam pipes, and leather aprons",
"a sunny classroom with a chalkboard, wooden desks, and colorful maps",
"a serene monastery courtyard with a stone fountain and orange trees",
"a dark noir alleyway with a single flickering streetlamp and brick walls",
"a vibrant Studio Ghibli-style meadow with rolling green hills and puffy clouds",
"a cluttered alchemy shop with jars of eyeballs and dried herbs",
"a sleek underground bunker with concrete walls and server racks",
"a majestic throne room with a red carpet and tall stained-glass windows",
"a peaceful park bench under a blooming cherry blossom tree",
"a retro-futuristic living room with bubble chairs and orange decor",
"a crowded subway car with handles hanging from the ceiling and ads",
"a desert oasis with a small pool of water and three tall palm trees",
"a messy garage with a vintage motorcycle and pegboards for tools",
"a floating island in the sky with a waterfall spilling into the clouds",
"a grand library with floor-to-ceiling shelves and rolling ladders",
"a cozy campsite with a tent, a log fire, and a starry night sky",
"a bright laundry room with white machines and stacks of wicker baskets",
"a neon-lit ramen shop with steam rising from bowls and wooden stools",
"a colorful circus tent interior with a center ring and red seating",
"a weathered lighthouse lantern room with a giant Fresnel lens",
"a futuristic greenhouse on Mars with red dust outside the glass dome",
"a traditional English pub with dark wood walls and brass rails",
"a magical floating marketplace with stalls tethered to hot air balloons",
"a dusty antique shop filled with clocks, porcelain dolls, and gramophones",
"a sleek penthouse balcony with an infinity pool and city views",
"a child's playroom with a colorful rug, toy blocks, and a teepee",
"a quiet museum gallery with marble statues and spotlighted paintings",
"a sun-drenched vineyard with rows of grapes and a stone villa",
"a dark gothic cathedral with pointed arches and flickering candles",
"a suburban backyard with a wooden fence, a grill, and a sliding set",
"a vibrant coral cave with glowing jellyfish and buried treasure chests",
"a futuristic medbay with white pods and scanning lasers",
"a cozy mountain ski lodge with a mudroom for boots and skis",
"a busy airport terminal with large windows and departure boards",
"a tranquil bamboo forest with a winding stone path",
"a colorful Mexican plaza with papel picado banners and a fountain",
"a hidden basement speakeasy with velvet curtains and a jazz stage",
"a majestic ice palace with walls of translucent blue ice",
"a cluttered jewelry maker's bench with pliers, wire, and gemstones",
"a peaceful rooftop with a telescope and a view of the Milky Way",
"a lively night market with hanging lightbulbs and street food carts",
"a serene botanical garden with a glass ceiling and lily pads",
"a high-fantasy dragon's lair with piles of gold coins and jewelry",
"a retro laundromat with rows of washers and checkered floors",
"a cozy nursery with a white crib, a mobile, and soft pastel walls",
"a sprawling junkyard with towers of scrap metal and rusted pipes",
"a peaceful autumn path covered in crunchy red and orange leaves",
"a sleek yacht deck with white leather sofas and a glass of champagne",
"a magical school hallway with moving portraits and stone arches",
"a futuristic control center with massive screens and swivel chairs",
"a messy pottery studio with a wheel and drying clay pots",
"a vibrant carnival at night with a glowing Ferris wheel and popcorn stands",
"a serene arctic landscape with an igloo and the northern lights",
"a cluttered sewing room with spools of thread and a dress mannequin",
"a grand opera house with red velvet seats and a gold-trimmed stage",
"a rustic barn interior with hay bales and hanging lanterns",
"a sleek hotel lobby with a marble floor and a minimalist front desk",
"a peaceful wildflower meadow with a wooden fence and a blue bird",
"a dark boiler room with rusted pipes, valves, and hissing steam",
"a cozy bakery with trays of croissants and the smell of fresh bread",
"a futuristic city park with holographic trees and floating benches",
"a traditional Japanese onsen with steam rising from a stone bath",
"a minimalist bedroom with a low bed and a single large window",
"a high-tech cybercafe with glowing blue keyboards and energy drink cans",
"a mystical library where books fly between shelves on their own",
"a cozy treehouse interior with a rope ladder and a telescope",
"a bustling fish market with wooden crates and crushed ice",
"a serene monastery library with ancient scrolls and candlelight",
"a futuristic train station with sleek maglev rails and glass pods",
"a messy science lab with chalkboard equations and a Tesla coil",
"a quiet cemetery in winter with snow-covered angel statues",
"a vibrant surf shop with colorful boards and hanging wetsuits",
"a dark ritual chamber with a stone altar and purple flames",
"a sunny conservatory with wicker furniture and hanging ferns",
"a classic detective's office with a ceiling fan and window blinds",
"a hidden forest shrine with stone lanterns and red torii gates",
"a sleek underground parking garage with concrete pillars and oil stains",
"a magical greenhouse with giant carnivorous plants and glowing spores",
"a medieval village tavern with heavy wooden mugs and a dirt floor",
"a futuristic apartment with a floor-to-ceiling view of a neon city",
"a quiet lakeside dock with a wooden rowing boat and a fishing rod",
"a cluttered watchmaker's shop with hundreds of ticking gears",
"a bright yoga studio with bamboo floors and rolled-up mats",
"a spooky abandoned asylum corridor with peeling paint and gurneys",
"a majestic mountain pass with a narrow stone bridge and high winds",
"a colorful candy shop with jars of rainbow gummies and lollipops",
"a high-fantasy elven city with bridges made of living tree roots",
"a retro recording studio with vintage microphones and sound foam",
"a serene desert canyon with orange rock walls and a thin stream",
"a bustling airport hangar with a half-assembled jet engine",
"a cozy attic bedroom with sloped ceilings and a circular window",
"a futuristic bioluminescent forest with glowing blue mushrooms",
"a dark limestone cave with massive stalactites and a hidden lake",
"a vibrant toy store with a giant floor piano and plush animals",
"a quiet suburban street at night with glowing porch lights",
"a sleek corporate rooftop bar with a fire pit and white sofas",
"a magical blacksmith forge with an anvil and sparks of blue light",
"a rustic winery cellar with rows of aging oak barrels",
"a bright fashion designer’s studio with fabric rolls and mannequins",
"a futuristic hospital ward with holographic vitals and floating beds",
"a serene koi pond with water lilies and a small red bridge",
"a cluttered comic book shop with long boxes and superhero posters",
"a dark medieval dungeon with iron shackles and a single torch",
"a sunny porch with a porch swing, lemonade, and a ceiling fan",
"a vibrant carnival game booth with stuffed prizes and bright lights",
"a futuristic server farm with rows of blinking green lights",
"a quiet mountain temple with a large bronze bell and incense",
"a messy architect's desk with blueprints, rulers, and coffee cups",
"a bright juice bar with baskets of wheatgrass and glass blenders",
"a spooky old clock tower with massive rusted bronze gears",
"a serene meadow at twilight with thousands of flickering fireflies",
"a sleek futuristic kitchen with hidden appliances and white marble",
"a rustic pottery shed with a wooden wheel and shelves of unglazed bowls",
"a vibrant underwater sunken city with merfolk and coral towers",
"a quiet library archive with rolling shelves and yellowed maps",
"a sunny classroom with small chairs and alphabet posters",
"a dark alleyway with steam rising from a manhole cover",
"a high-fantasy throne room made entirely of obsidian and gold",
"a futuristic hydroponic farm with purple lights and vertical pipes",
"a cozy tea cottage with floral wallpaper and lace tablecloths",
"a bustling construction site with yellow cranes and steel beams",
"a serene winter forest with a frozen waterfall and pine trees",
"a cluttered florist shop with buckets of roses and green shears",
"a sleek modern museum hall with a single minimalist sculpture",
"a magical floating garden with islands held up by golden chains",
"a dark stormy beach with crashing waves and jagged black rocks",
"a bright photography studio with white backdrops and softbox lights",
"a futuristic bridge of a starship with a view of a swirling nebula",
"a cozy knitting nook with baskets of yarn and a rocking chair",
"a vibrant street festival with paper lanterns and food stalls",
"a quiet zen rock garden with a small bamboo water fountain",
"a messy carpentry workshop with piles of sawdust and hand saws",
"a high-fantasy dragon's nest with large translucent blue eggs",
"a futuristic gym with holographic trainers and neon weight racks",
"a serene cherry blossom path with fallen pink petals on the ground",
"a dark Victorian parlor with velvet wallpaper and a gramophone",
"a bright breakfast nook with a round table and a vase of daisies",
"a spooky forest path with twisted trees that look like arms",
"a sleek minimalist office with a single plant and a laptop",
"a magical potion shop with bubbling cauldrons and glass vials",
"a vibrant soccer stadium with green turf and bright floodlights",
"a quiet monastery cell with a wooden cross and a stone bed",
"a cluttered basement workshop with a workbench and old radios",
"a high-fantasy floating castle surrounded by white clouds",
"a futuristic pod hotel corridor with small circular doors",
"a serene autumn lake with a wooden cabin and smoke from the chimney",
"a bright art gallery opening with people holding wine glasses",
"a dark abandoned mall with broken mannequins and vines",
"a cozy cat cafe with scratching posts and sleeping kittens",
"a vibrant coral reef with a giant sea turtle and colorful sponges",
"a quiet snowy street with a single red mailbox and a streetlamp",
"a sleek modern fireplace in a room with floor-to-ceiling glass",
"a magical ice cave with frozen flowers and blue light",
"a bustling ramen stall in a rainy neon-lit alleyway",
"a cluttered jewelry workshop with a magnifying glass and gold wire",
"a high-fantasy forest village with houses built into giant oaks",
"a futuristic laboratory with a large holographic DNA strand",
"a serene desert dunes under a vast starry night sky",
"a bright sunroom with a hammock and many potted succulents",
"a dark mahogany-paneled study with a green banker’s lamp",
"a cozy mountain hut with a wood-burning stove and wool blankets",
"a vibrant flower market with buckets of sunflowers and tulips",
"a sleek futuristic elevator with a glass floor and city views"
]

### Step 5: Run Dataset Generation
Define your target output directory inside your Google Drive and run the generation loop. Each scene defined above will generate `num_images_per_scene` images and save them directly to your Drive.

In [ ]:
import os

output_dir = "/content/drive/MyDrive/generated_dataset"
os.makedirs(output_dir, exist_ok=True)

num_images_per_scene = 1  # adjustable

for i, scene in enumerate(scenes):
    prompt = BASE_PROMPT.format(scene=scene)

    for j in range(num_images_per_scene):
        image = pipe(
            prompt=prompt,
            num_inference_steps=4,
            guidance_scale=0.0).images[0]

        filename = f"{output_dir}/img_{i}_{j}.png"
        image.save(filename)

        print(f"Saved {filename}")

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['2 4']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['2 4']


  0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['4']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['4']


Saved /content/drive/MyDrive/generated_dataset/img_536_0.png


  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['2 4']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['2 4']


Saved /content/drive/MyDrive/generated_dataset/img_644_0.png


  0%|          | 0/4 [00:00<?, ?it/s]

Saved /content/drive/MyDrive/generated_dataset/img_645_0.png
